In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/D-alanine_60K_278466_c-rot_unopt_rPBE_SEDC_magres.magres"


This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file

*Make sure you change the nucleus information like Q value and nucleus (I am not sure how to do this in an efficient way)*

Shiva Agarwal

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'O'      # nucleus for which parameters are wanted
atom_label = 4      # site for which parameters wanted
Q = -0.0256         #electric quadrupole moment for nucleus in barn

In [7]:
for atom in atoms.species(nucleus):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[ -43.2580826   204.21797039  -59.93603151]
 [ 175.66382911  128.39154553   84.23728864]
 [ -16.90039422   43.60931701 -158.47313467]]

17O2 sigma:
 [[ -43.2580826  -204.21797039   59.93603151]
 [-175.66382911  128.39154553   84.23728864]
 [  16.90039422   43.60931701 -158.47313467]]

17O3 sigma:
 [[ -43.2580826   204.21797039   59.93603151]
 [ 175.66382911  128.39154553  -84.23728864]
 [  16.90039422  -43.60931701 -158.47313467]]

17O4 sigma:
 [[ -43.2580826  -204.21797039  -59.93603151]
 [-175.66382911  128.39154553  -84.23728864]
 [ -16.90039422  -43.60931701 -158.47313467]]

17O5 sigma:
 [[-57.67760658 167.99280046  77.88159431]
 [171.12053154  88.44626679 -26.52101791]
 [ -9.76702765  24.62086332 -47.01692136]]

17O6 sigma:
 [[ -57.67760658 -167.99280046  -77.88159431]
 [-171.12053154   88.44626679  -26.52101791]
 [   9.76702765   24.62086332  -47.01692136]]

17O7 sigma:
 [[-57.67760658 167.99280046 -77.88159431]
 [171.12053154  88.44626679  26.52101791]
 [  9.767027

In [8]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 8.356408181027579

17O2 sigma:
 8.356408181027598

17O3 sigma:
 8.356408181027609

17O4 sigma:
 8.356408181027602

17O5 sigma:
 6.736217320324397

17O6 sigma:
 6.736217320324395

17O7 sigma:
 6.736217320324449

17O8 sigma:
 6.736217320324449



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.944 -0.938 -4.904]
 [-0.938 -0.784  3.712]
 [-4.904  3.712 -0.16 ]]

CS Tensor:
 [[-57.678 167.993  77.882]
 [171.121  88.446 -26.521]
 [ -9.767  24.621 -47.017]]

CS isotropic Tensor:
 [[-5.416  0.     0.   ]
 [ 0.    -5.416  0.   ]
 [ 0.     0.    -5.416]]

CS symmetric Tensor:
 [[-57.678 169.557  34.057]
 [169.557  88.446  -0.95 ]
 [ 34.057  -0.95  -47.017]]

CS antisymmetric Tensor:
 [[  0.     -1.564  43.824]
 [  1.564   0.    -25.571]
 [-43.824  25.571   0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.7414818  -1.04917018 -5.69231161] 

 Unsorted Eigenvectors:
 [[-0.6276618   0.6118595   0.48131965]
 [ 0.40589463  0.78479752 -0.46834004]
 [ 0.66429678  0.09859409  0.74093792]] 

Sorted Eigenvalues: 
 [-1.04917018 -5.69231161  6.7414818 ] 

Sorted Eigenvectors: 
 [[ 0.6118595   0.48131965 -0.6276618 ]
 [ 0.78479752 -0.46834004  0.40589463]
 [ 0.09859409  0.74093792  0.66429678]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 201.31834282 -175.8052347   -41.76136927] 

 Unsorted Eigenvectors:
 [[ 0.55295433  0.8204949   0.14501596]
 [ 0.83003755 -0.52726368 -0.18174344]
 [ 0.07265791 -0.22086452  0.97259431]] 

Sorted Eigenvalues: 
 [ -41.76136927 -175.8052347   201.31834282] 

Sorted Eigenvectors: 
 [[ 0.14501596  0.8204949   0.55295433]
 [-0.18174344 -0.52726368  0.83003755]
 [ 0.97259431 -0.22086452  0.07265791]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -1.0491701810136767 -5.692311614024033 6.741481795037749
CSA Tensor Components δyy, δxx, δzz: 
 -41.76136926699313 -175.8052346987906 201.31834281544633


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 17O5: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   6.74148  |
+--------------+------------+
| etaq         |   0.688742 |
+--------------+------------+
| iso_cs (ppm) |  -5.41609  |
+--------------+------------+
| csa (ppm)    | 206.734    |
+--------------+------------+
| etas         |   0.648387 |
+--------------+------------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/O_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.48131965  0.6118595  -0.6276618 ]
 [-0.46834004  0.78479752  0.40589463]
 [ 0.74093792  0.09859409  0.66429678]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
7.5796274124807885 48.37160129379061 32.889831158437694 

Direction cosine csa: 

[[ 0.8204949   0.14501596  0.55295433]
 [-0.52726368 -0.18174344  0.83003755]
 [-0.22086452  0.97259431  0.07265791]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-77.2057974872352 85.8333365627122 -56.329245306396466 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -28.793314501459957 chi: 87.81616444185981 xi: 86.05453143327648 

